### LLM : 질의응답 챗봇 개발 파이프라인

- 이 노트북은 **`질의 응답 챗봇` 개발 파이프라인 예제 실습**을 수행하는 노트북입니다.

##### Llama를 활용한 도메인 특화 챗봇 개발 파이프라인 예제
   1.  Llama를 활용한 질의응답 챗봇 Fine-Tuning 및 성능 평가 (Cell 6개)

### Llama를 활용한 질의응답 챗봇 Fine-Tuning 및 성능 평가
사전학습된 Llama 3.2 Korean Bllossom 모델을 KoAlpaca 데이터셋으로 Fine-tuning하여 한국어 질의응답 성능을 향상시킵니다.

LoRA(Low-Rank Adaptation) 기법을 활용하여 효율적인 파라미터 튜닝을 수행하고 ROUGE 및 F1 Score로 성능을 정량적으로 평가합니다.

* KoAlpaca 데이터셋 로드 및 한국어 텍스트 전처리로 instruction-input-output 구조를 질의응답 형태로 변환
* Bllossom Llama 3.2 3B 모델을 4-bit 양자화로 로드하고 메모리 효율성 확보
* 대화형 메시지 포맷으로 데이터 변환하여 시스템-사용자-어시스턴트 역할 구조화
* LoRA 설정으로 어텐션 모듈만 선택적 학습하여 계산 비용 절감 및 과적합 방지
* SFTTrainer를 사용한 지도학습 Fine-tuning 실행 및 학습 과정 모니터링
* ROUGE-1/2/L 및 F1 Score 기반 정량적 성능 평가와 샘플 예측 결과 확인

In [1]:
# ============================================ 
# Cell 1: 환경 설정 및 라이브러리 임포트 
# ============================================

import torch
import transformers
import peft
import os
import warnings
warnings.filterwarnings('ignore')  # 경고 메시지 숨김

# 현재 환경 정보 출력
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"GPU 이름: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"Transformers 버전: {transformers.__version__}")
print(f"PEFT 버전: {peft.__version__}")

# GPU 연산 속도 최적화 설정
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # CUDA 커널 비동기 실행
torch.backends.cuda.matmul.allow_tf32 = True    # TensorFloat-32 연산 활성화
torch.backends.cudnn.allow_tf32 = True          # cuDNN TF32 연산 활성화
torch.backends.cudnn.benchmark = True           # 최적 알고리즘 자동 선택

c:\Users\SSAFY\.conda\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 버전: 2.7.1+cu128
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 4050 Laptop GPU
GPU 메모리: 6.00 GB
Transformers 버전: 4.46.3
PEFT 버전: 0.12.0


In [2]:
# ============================================ 
# Cell 2: 데이터셋 로드 및 전처리 
# ============================================

from datasets import load_dataset
import re
from typing import Dict 

# KoAlpaca 한국어 instruction 데이터셋 로드
dataset = load_dataset("Taegyuu/KoAlpaca-v1.1a", trust_remote_code=True)

# 데이터셋 기본 정보 확인
print(f"데이터셋 로드 완료!")
print(f"학습 데이터: {len(dataset['train'])} 샘플")
print(f"\n데이터셋 구조:")
print(dataset)

# 첫 번째 샘플의 구조 분석
print("\n샘플 데이터 확인:")
sample = dataset['train'][0]
print(sample)
# 각 필드의 내용 미리보기 (100자 제한)
for key in sample.keys():
    print(f"{key}: {str(sample[key])[:100]}...")

def preprocess_korean_text(text: str) -> str:
    """한국어 텍스트 정리 및 정규화"""
    # HTML/JavaScript 코드 완전 제거
    text = re.sub(r'<script.*?</script>', '', text, flags=re.DOTALL)
    text = re.sub(r'<style.*?</style>', '', text, flags=re.DOTALL)
    
    # HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)
    
    # HTML 특수문자 변환
    text = text.replace('&nbsp;', ' ')
    text = text.replace('&lt;', '<')
    text = text.replace('&gt;', '>')
    text = text.replace('&amp;', '&')
    text = text.replace('&quot;', '"')
    
    # 한글, 영문, 숫자, 기본 문장부호만 유지
    text = re.sub(r'[^\w\s가-힣.,!?0-9%]', ' ', text)
    
    # 연속된 공백을 하나로 통합
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def process_koalpaca_example(example: Dict) -> Dict:
    """KoAlpaca 형식을 표준 QA 형식으로 변환"""
    # instruction, input, output 필드 전처리
    instruction = preprocess_korean_text(example.get('instruction', ''))
    input_text = preprocess_korean_text(example.get('input', ''))
    output = preprocess_korean_text(example.get('output', ''))
    
    # instruction과 input을 결합하여 완전한 질문 생성
    if input_text.strip():
        question = f"{instruction}\n\n{input_text}"
    else:
        question = instruction
    
    # 표준 QA 형식으로 반환
    return {
        'id': f"koalpaca_{hash(instruction)}",  # 고유 ID 생성
        'context': '',  # KoAlpaca는 별도 context 없음
        'question': question,
        'answer': output
    }

# 전체 데이터셋 전처리 실행
print("데이터 전처리 중...")
processed_dataset = dataset.map(process_koalpaca_example)

# 학습 시간 단축을 위한 샘플 수 제한
max_train_samples = 1000  # 학습용 샘플 수
max_eval_samples = 100    # 검증용 샘플 수

if len(processed_dataset['train']) > max_train_samples:
    processed_dataset['train'] = processed_dataset['train'].select(range(max_train_samples))

# validation set이 없으므로 train에서 분할
print("Validation set이 없어서 train 데이터에서 분할합니다.")
total_samples = len(processed_dataset['train'])
train_end = total_samples - max_eval_samples

# 마지막 샘플들을 검증용으로 분리
processed_dataset['validation'] = processed_dataset['train'].select(range(train_end, total_samples))
processed_dataset['train'] = processed_dataset['train'].select(range(train_end))

print(f"전처리 완료!")
print(f"학습 데이터: {len(processed_dataset['train'])} 샘플")
print(f"검증 데이터: {len(processed_dataset['validation'])} 샘플")

# 전처리 결과 확인
print("\n전처리된 샘플:")
sample = processed_dataset['train'][0]
print(f"질문: {sample['question']}")
print(f"답변: {sample['answer']}")
print(f"문맥: {sample['context'][:100]}...")

print(f"\n데이터셋 컬럼: {processed_dataset['train'].column_names}")
print(f"데이터셋 특징: {processed_dataset['train'].features}")

def format_conversational_data(message):
    """표준 QA 형식을 Llama 대화 형식으로 변환"""
    question = message['question']  
    context = message.get('context', None)
    answer = message.get('answer', None)

    # 시스템 프롬프트 정의
    system_message = "당신은 한국어 질의응답 전문가입니다. 주어진 문맥을 바탕으로 정확하고 간결한 답변을 제공하세요."
    
    # 문맥 유무에 따른 사용자 메시지 구성
    if context and context.strip():
        user_message = f"다음 문서를 참고하여 질문에 답변해주세요.\n\n[참고 문서]\n{context}\n\n[질문]\n{question}"
    else:
        user_message = question

    # 학습용: 답변 포함된 완전한 대화
    if answer:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer}
        ]
    # 추론용: 답변 없는 대화 (생성 대기)
    else:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": [{"type": "text", "text": user_message}]}
        ]
    
    return {"messages": messages}

# 학습용 대화 형식 변환
print("데이터셋을 conversational 형식으로 변환 중...")
train_dataset = processed_dataset['train'].map(format_conversational_data,
    remove_columns=processed_dataset['train'].column_names)
eval_dataset = processed_dataset['validation'].map(format_conversational_data,
    remove_columns=processed_dataset['validation'].column_names)

print("변환 완료!")
print(f"최종 학습 데이터: {len(train_dataset)} 샘플")
print(f"최종 검증 데이터: {len(eval_dataset)} 샘플")

# 최종 변환 결과 확인
print("\n=== 최종 샘플 확인 ===")
final_sample = train_dataset[0]
print("Messages:")
for i, msg in enumerate(final_sample['messages']):
    print(f"  {i+1}. {msg['role']}: {msg['content'][:150]}{'...' if len(msg['content']) > 150 else ''}")


데이터셋 로드 완료!
학습 데이터: 21155 샘플

데이터셋 구조:
DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 21155
    })
})

샘플 데이터 확인:
{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?', 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268'}
instruction: 양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?...
output: 양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 

식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파...
url: https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268...
데이터 전처리 중...
Validation set이 없어서 train 데이터에서 분할합니다.
전처리 완료!
학습 데이터: 900 샘플
검증 데이터: 100 샘플

In [3]:
# ============================================ 
# Cell 3: 모델 및 토크나이저 로드 
# ============================================

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 한국어 특화 Llama 모델 선택
model_name = "Bllossom/llama-3.2-Korean-Bllossom-3B"
print(f"선택된 모델: {model_name}")

# 메모리 절약을 위한 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 양자화 활성화
    bnb_4bit_quant_type="nf4",           # NormalFloat4 양자화 타입
    bnb_4bit_compute_dtype=torch.float16, # 연산용 데이터 타입
    bnb_4bit_use_double_quant=True,      # 이중 양자화로 메모리 추가 절약
)

# 모델 로드 (양자화 적용)
print("모델 로딩 중... (몇 분 소요될 수 있습니다)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,      # 양자화 설정 적용
    device_map="auto",                   # 자동 GPU 할당
    torch_dtype=torch.float16,           # 반정밀도 부동소수점
    trust_remote_code=True,              # 사용자 정의 코드 신뢰
    offload_folder="./offload"
)

# 토크나이저 로드 및 설정
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # 패딩 토큰 설정
tokenizer.padding_side = "right"               # 오른쪽 패딩

print("모델 및 토크나이저 로드 완료!")
print(f"모델 크기: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B 파라미터")

# 초기 모델 성능 테스트
test_context = "서울특별시는 대한민국의 수도이며, 인구는 약 950만 명입니다."
test_question = "한국의 수도는 어디인가요?"
template = format_conversational_data({'context': test_context, 'question': test_question})
test_prompt = tokenizer.apply_chat_template( 
        template['messages'], 
        tokenize=False, 
        add_generation_prompt=True
    )

# 입력 토큰화 및 GPU 이동
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

print("초기 추론 테스트:")
print(f"질문: {test_question}")

# 테스트 추론 실행
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,                # 최대 생성 토큰 수
        temperature=0.7,                  # 생성 다양성 조절
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# 생성된 답변만 추출하여 출력
response = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"답변: {response}")

선택된 모델: Bllossom/llama-3.2-Korean-Bllossom-3B
모델 로딩 중... (몇 분 소요될 수 있습니다)


Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\SSAFY\.conda\envs\llm\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\SSAFY\.conda\envs\llm\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\SSAFY\.conda\envs\llm\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\SSAFY\.conda\envs\llm\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\Users\SSAFY\.conda\envs\llm\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.97s/it]


모델 및 토크나이저 로드 완료!
모델 크기: 1.80B 파라미터
초기 추론 테스트:
질문: 한국의 수도는 어디인가요?
답변: 서울입니다.


In [4]:
# ============================================ 
# Cell 4: LoRA Fine-tuning 설정 및 실행 
# ============================================

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

# LoRA(Low-Rank Adaptation) 파라미터 설정
lora_config = LoraConfig(
    r=32,                    # 저차원 행렬의 rank (클수록 표현력 증가)
    lora_alpha=64,           # 스케일링 팩터 (일반적으로 2*r)
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], # Llama 어텐션 모듈
    lora_dropout=0.05,       # 과적합 방지용 드롭아웃
    bias="none",             # 바이어스 학습 비활성화
    task_type="CAUSAL_LM"    # 인과적 언어모델 태스크
)

# Fine-tuning 하이퍼파라미터 설정
training_args = SFTConfig(
    output_dir="./korean-qa-lora",       # 모델 저장 경로
    num_train_epochs=1,                  # 학습 에포크 수
    per_device_train_batch_size=2,       # 훈련 배치 크기
    per_device_eval_batch_size=2,        # 평가 배치 크기
    gradient_checkpointing=True,         # 메모리 절약을 위한 체크포인팅
    gradient_accumulation_steps=4,       # 그래디언트 누적 단계
    optim="paged_adamw_8bit",           # 8-bit AdamW 옵티마이저
    logging_steps=10,                    # 로깅 주기
    logging_first_step=True,             # 첫 스텝 로깅
    logging_strategy="steps",            # 스텝 기반 로깅
    learning_rate=2e-4,                  # 학습률
    warmup_steps=100,                    # 워밍업 스텝
    save_strategy="steps",               # 모델 저장 전략
    save_steps=200,                      # 저장 주기
    eval_strategy="steps",               # 평가 전략
    eval_steps=50,                       # 평가 주기
    lr_scheduler_type="cosine",          # 코사인 학습률 스케줄러
    fp16=True,                           # 16-bit 부동소수점 연산
    max_seq_length=512,                  # 최대 시퀀스 길이
    packing=False,                       # 시퀀스 패킹 비활성화
    report_to="none",                    # 외부 로깅 도구 사용 안함
    max_grad_norm=1.0,                   # 그래디언트 클리핑
    seed=42,                             # 재현 가능성을 위한 시드
    dataloader_num_workers=4,            # 데이터 로더 워커 수
    dataloader_pin_memory=True,          # 메모리 핀 고정
    dataset_num_proc=4,                  # 데이터 전처리 병렬화
    neftune_noise_alpha=5,               # NEFTune 노이즈로 학습 안정성 향상
)

print("학습 설정 완료!")

# 데이터셋 샘플 최종 확인
print("학습 데이터셋 최종 샘플 출력")
print(train_dataset[0])
print(eval_dataset[0])

# SFTTrainer 초기화 (Supervised Fine-Tuning)
trainer = SFTTrainer(
    model=model,                         # 학습할 모델
    args=training_args,                  # 학습 인자
    train_dataset=train_dataset,         # 훈련 데이터
    eval_dataset=eval_dataset,           # 검증 데이터
    tokenizer=tokenizer,                 # 토크나이저
    peft_config=lora_config,             # LoRA 설정
)

print("Fine-tuning 시작!")

# 실제 Fine-tuning 실행
trainer.train()

print("Fine-tuning 완료!")
# 학습 결과 요약 출력
print(trainer.state.log_history[-1])  # 마지막 로그
losses = [log['loss'] for log in trainer.state.log_history if 'loss' in log]
print(f"Loss 변화: {losses[0]} → {losses[-1]}")

# 학습된 모델 저장
trainer.save_model("./korean-qa-lora")
tokenizer.save_pretrained("./korean-qa-lora")

print("모델 저장 완료!")
print("저장 위치: ./korean-qa-lora")

학습 설정 완료!
학습 데이터셋 최종 샘플 출력
{'messages': [{'content': '당신은 한국어 질의응답 전문가입니다. 주어진 문맥을 바탕으로 정확하고 간결한 답변을 제공하세요.', 'role': 'system'}, {'content': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?', 'role': 'user'}, {'content': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다. 덧붙이는 답변 고구마 줄기도 볶아먹을 수 있나요? 고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.', 'role': 'assistant'}]}
{'messages': [{'content': '당신은 한국어 질의응답 전문가입니다. 주어진 문맥을 바탕으로 정확하고 간결한 답변을 제공하세요.', 'role': 'system'}, {'content': '일본의 일촌일품이란 정책은 무엇인가요?', 'role': 'user'}, {'content': '일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이

  1%|          | 1/112 [00:52<1:37:52, 52.90s/it]

{'loss': 2.9707, 'grad_norm': 2.426041841506958, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.01}


  9%|▉         | 10/112 [05:48<50:36, 29.77s/it] 

{'loss': 2.8811, 'grad_norm': 1.401610016822815, 'learning_rate': 2e-05, 'epoch': 0.09}


...

{'eval_loss': 1.968306064605713, 'eval_runtime': 134.5613, 'eval_samples_per_second': 0.743, 'eval_steps_per_second': 0.372, 'epoch': 0.89}


 98%|█████████▊| 110/112 [52:01<00:49, 24.59s/it]

{'loss': 1.9281, 'grad_norm': 0.9652595520019531, 'learning_rate': 1.339745962155613e-05, 'epoch': 0.98}


100%|██████████| 112/112 [52:57<00:00, 28.37s/it]


{'train_runtime': 3177.8429, 'train_samples_per_second': 0.283, 'train_steps_per_second': 0.035, 'train_loss': 2.144712431090219, 'epoch': 1.0}
Fine-tuning 완료!
{'train_runtime': 3177.8429, 'train_samples_per_second': 0.283, 'train_steps_per_second': 0.035, 'total_flos': 5723957408108544.0, 'train_loss': 2.144712431090219, 'epoch': 0.9955555555555555, 'step': 112}
Loss 변화: 2.9707 → 1.9281
모델 저장 완료!
저장 위치: ./korean-qa-lora


In [5]:
# ============================================ 
# Cell 5: 답변 생성 함수 정의 
# ============================================

def generate_answer(question: str, context: str, max_length: int = 100) -> str:
    """Fine-tuning된 모델로 질문에 대한 답변 생성"""
    # 입력을 대화 형식으로 변환
    template = format_conversational_data({'context': context, 'question': question})
    prompt = tokenizer.apply_chat_template(
        template['messages'], 
        tokenize=False, 
        add_generation_prompt=True  # 어시스턴트 응답 시작 토큰 추가
    )
    
    # 프롬프트 토큰화 및 길이 제한
    inputs = tokenizer(
        text=prompt,
        return_tensors="pt", 
        truncation=True, 
        max_length=1024
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # 텍스트 생성 파라미터 설정
    generation_config = {
        'max_new_tokens': max_length,        # 최대 생성 토큰 수
        'repetition_penalty': 1.2,           # 반복 방지 페널티
        'no_repeat_ngram_size': 3,           # n-gram 반복 방지
        # 평가를 위한 결정적 생성 설정
        'do_sample': False,                  # 샘플링 비활성화
        'num_beams': 4,                      # 빔 서치 사용
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
    }
    
    # 답변 생성 실행
    with torch.no_grad():
        outputs = model.generate(**inputs, **generation_config)
    
    # 새로 생성된 토큰만 디코딩
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # 답변 후처리: 중복 문장 제거 및 정리
    response = response.strip()
    sentences = response.split('.')
    unique_sentences = []
    for sent in sentences:
        if sent.strip() and sent.strip() not in unique_sentences:
            unique_sentences.append(sent.strip())
    response = '. '.join(unique_sentences)
    if response and not response.endswith('.'):
        response += '.'
    
    return response

In [6]:
# ============================================ 
# Cell 6: 성능 평가 메트릭 및 모델 평가 
# ============================================

from collections import Counter
import numpy as np
import random
from tqdm import tqdm
from rouge_score import rouge_scorer

def normalize_answer(text: str) -> str:
    """답변 텍스트 정규화 (대소문자, 공백, 특수문자 처리)"""
    text = text.lower()                                      # 소문자 변환
    text = re.sub(r'\s+', ' ', text)                        # 연속 공백 제거
    text = re.sub(r'[^\w\s가-힣0-9]', '', text)               # 특수문자 제거
    return text.strip()

def compute_f1_score(prediction: str, ground_truth: str) -> float:
    """토큰 레벨 F1 Score 계산"""
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    
    # 빈 예측이나 정답 처리
    if not pred_tokens or not truth_tokens:
        return 0.0
    
    # 공통 토큰 개수 계산
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    # Precision, Recall, F1 계산
    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    
    return f1

def compute_rouge_scores(prediction: str, ground_truth: str) -> dict:
    """ROUGE-1, ROUGE-2, ROUGE-L 점수 계산"""
    print(f"\n예측 원본: {prediction} / 정규화: {normalize_answer(prediction)}")
    print(f"\n정답 원본: {ground_truth} / 정규화: {normalize_answer(ground_truth)}")
    
    # ROUGE 스코어러 초기화 (한국어는 stemmer 비활성화)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    scores = scorer.score(normalize_answer(ground_truth), normalize_answer(prediction))
    
    return {
        'rouge1': scores['rouge1'].fmeasure,   # 1-gram F1
        'rouge2': scores['rouge2'].fmeasure,   # 2-gram F1  
        'rougeL': scores['rougeL'].fmeasure    # LCS F1
    }

print("평가 메트릭 정의 완료!")

# 평가용 샘플 선택 (계산 시간 고려)
eval_samples = processed_dataset['validation'].select(range(min(10, len(processed_dataset['validation']))))

# 평가 점수 저장 리스트
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
f1_scores = []

print("모델 평가 중...")
# 각 샘플에 대해 예측 및 평가 수행
for sample in tqdm(eval_samples):
    print(sample)
    # 모델 예측 생성
    prediction = generate_answer(sample['question'], sample['context'], max_length=256)
    ground_truth = sample['answer']
    
    # 각종 메트릭 계산
    rouge_scores = compute_rouge_scores(prediction, ground_truth)
    f1 = compute_f1_score(prediction, ground_truth)
    
    # 점수 리스트에 추가
    rouge1_scores.append(rouge_scores['rouge1'])
    rouge2_scores.append(rouge_scores['rouge2'])
    rougeL_scores.append(rouge_scores['rougeL'])
    f1_scores.append(f1)

# 평균 성능 계산 및 출력
avg_rouge1 = np.mean(rouge1_scores) * 100
avg_rouge2 = np.mean(rouge2_scores) * 100
avg_rougeL = np.mean(rougeL_scores) * 100
avg_f1 = np.mean(f1_scores) * 100

print(f"\n평가 결과:")
print(f"F1 Score: {avg_f1:.2f}%")
print(f"ROUGE-1: {avg_rouge1:.2f}%")
print(f"ROUGE-2: {avg_rouge2:.2f}%")
print(f"ROUGE-L: {avg_rougeL:.2f}%")

# 모델을 평가 모드로 전환
model.eval()

# 실제 예측 결과 샘플 확인
print("\n예측 샘플:")
for i in range(3):
    sample = eval_samples[i]
    prediction = generate_answer(sample['question'], sample['context'], max_length=256)
    print(f"\n질문: {sample['question']}")
    print(f"정답: {sample['answer']}")
    print(f"예측: {prediction}")

# GPU 메모리 정리
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
    
torch.cuda.empty_cache()  # GPU 캐시 비우기

print("GPU 메모리 정리 완료!")
print(f"현재 GPU 메모리 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

평가 메트릭 정의 완료!
모델 평가 중...


  0%|          | 0/10 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


{'instruction': '일본의 일촌일품이란 정책은 무엇인가요?', 'output': '일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이를 흉내내어 우리나라 지자체도 지역경제 활성화를 위해 다양한 축제와 행사를 개최하는 경우가 많습니다. 하지만 일촌일품이 실패하는 경우 지자체는 부채에 시달리는 경우도 있으므로 신중한 기획과 검토가 필요합니다.', 'url': 'https://kin.naver.com/qna/detail.naver?d1id=9&dirId=9020101&docId=74933167', 'id': 'koalpaca_3619485372677877410', 'context': '', 'question': '일본의 일촌일품이란 정책은 무엇인가요?', 'answer': '일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이를 흉내내어 우리나라 지자체도 지역경제 활성화를 위해 다양한 축제와 행사를 개최하는 경우가 많습니다. 하지만 일촌일품이 실패하는 경우 지자체는 부채에 시달리는 경우도 있으므로 신중한 기획과 검토가 필요합니다.'

 10%|█         | 1/10 [04:33<40:58, 273.19s/it]


예측 원본: 일본은 1951년 4월 1일부터 1965년 2월 28일까지 14년간 이탈리아, 프랑스, 스페인, 벨기에, 네덜란드, 오스트리아, 스위스, 스웨덴, 노르웨이, 덴마크, 이스라엘, 이집트, 모로코, 이란, 이라크, 알제리, 튀르키예, 인도, 파키스탄, 베트남, 태국, 말레이시아, 싱가포르, 필리핀, 뉴질랜드, 아르헨티나, 우루과이, 브라질, 페루, 칠레, 콜롬비아, 에콰도르, 베네수엘라, 파라과이 등 16개국에 대한 일촌을 제공하는 정책입니다. 이 정책은 일본이 1950년대 후반부터 60년대 초반까지 수출을 확대할 수 있도록 하기 위해 추진되었습니다. 일본은 이 정책을 통해 수출량을 10배 이상 증가시켰습니다. 또한, 이 정책으로 인. / 정규화: 일본은 1951년 4월 1일부터 1965년 2월 28일까지 14년간 이탈리아 프랑스 스페인 벨기에 네덜란드 오스트리아 스위스 스웨덴 노르웨이 덴마크 이스라엘 이집트 모로코 이란 이라크 알제리 튀르키예 인도 파키스탄 베트남 태국 말레이시아 싱가포르 필리핀 뉴질랜드 아르헨티나 우루과이 브라질 페루 칠레 콜롬비아 에콰도르 베네수엘라 파라과이 등 16개국에 대한 일촌을 제공하는 정책입니다 이 정책은 일본이 1950년대 후반부터 60년대 초반까지 수출을 확대할 수 있도록 하기 위해 추진되었습니다 일본은 이 정책을 통해 수출량을 10배 이상 증가시켰습니다 또한 이 정책으로 인

정답 원본: 일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이를 흉내내어 우리나라 지자체도 지역경제 활성화를 위해 다양한 축제

...

 90%|█████████ | 9/10 [1:01:32<08:31, 512.00s/it]


예측 원본: 금은 금속이기 때문에 얼리는 것은 불가능합니다. 하지만, 금속의 특성상 부식이나 손실이 발생할 수 있습니다. 부식이 발생하는 이유는 여러 가지가 있지만, 가장 일반적인 이유는 금속 표면의 오염으로 인해 발생합니다. 오염이 생기는 경우에는 금속을 청소하여 제거하는 것이 좋습니다. 또한, 금을 사용하는 장비나 도구를 정기적으로 청소하는 것이 중요합니다. 금은 물에 잘 녹지 않기 때문에 물에 담그지 않도록 주의해야 합니다. 금이 손상된 경우에는 전문가의 도움이 필요할 수 있으므로 조심스럽게 다루어야 합니다. 따라서, 금은 얼리는 것이 아니라, 부식과 손실을 예방하기 위해 적절한 관리와 청소가 필요합니다. 추가 정보 금이 얼리는 것과 부식하는 것의 차이점은 무엇인가요? 금을 얼리는 건 없고, 금이 부식되는 건 무엇일까요? 금의 부식은 금의 표면이 오염되어 발생하는 현상입니다. 금을 부식시킬 수 있는 원. / 정규화: 금은 금속이기 때문에 얼리는 것은 불가능합니다 하지만 금속의 특성상 부식이나 손실이 발생할 수 있습니다 부식이 발생하는 이유는 여러 가지가 있지만 가장 일반적인 이유는 금속 표면의 오염으로 인해 발생합니다 오염이 생기는 경우에는 금속을 청소하여 제거하는 것이 좋습니다 또한 금을 사용하는 장비나 도구를 정기적으로 청소하는 것이 중요합니다 금은 물에 잘 녹지 않기 때문에 물에 담그지 않도록 주의해야 합니다 금이 손상된 경우에는 전문가의 도움이 필요할 수 있으므로 조심스럽게 다루어야 합니다 따라서 금은 얼리는 것이 아니라 부식과 손실을 예방하기 위해 적절한 관리와 청소가 필요합니다 추가 정보 금이 얼리는 것과 부식하는 것의 차이점은 무엇인가요 금을 얼리는 건 없고 금이 부식되는 건 무엇일까요 금의 부식은 금의 표면이 오염되어 발생하는 현상입니다 금을 부식시킬 수 있는 원

정답 원본: 금은 고체인 물질이므로 얼릴 수는 있지만, 얼려도 부식이나 손실은 발생하지 않습니다. 금은 온도가 내려가면 부피는 약간 수축할 수 있지만, 무게는 변하지 않습니다.

100%|██████████| 10/10 [1:06:54<00:00, 401.47s/it]


예측 원본: 어두울 때 잠이 오는 이유는 신체 내부의 호르몬 수치가 낮아지기 때문입니다. 호르โมन은 신체의 다양한 활동을 조절하는 역할을 하며, 어두운 환경에서는 신체가 활발한 활동을 할 수 없기 때문에 호르모ンの 수치는 낮아집니다. 또한, 어둠은 신경계의 활동을 억제시켜서 잠을 자도록 유도합니다. 새벽에는 기압의 변화로 인해 호흡과 혈액순환, 신체 활동 등이 활발하게 이루어지지 않기 때문에 잠이 자는 시간이 많습니다. 이러한 이유로 새벽에 자는 것이 일반적입니다. 하지만, 개인마다의 차이가 있을 수 있으므로, 개인적으로 적합한 잠의 시간을 맞추는 것이 좋습니다. 예를 들면, 일주일에 7 8시간 정도의 수면이 적절한 것으로 알려져 있습니다. 따라서, 개인적인 수면 습관을 조정하여 적정 수면량을 유지하는 것이 중요합니다. 이외에도, 수면 환경, 식사, 운동, 스트레스. / 정규화: 어두울 때 잠이 오는 이유는 신체 내부의 호르몬 수치가 낮아지기 때문입니다 호르โมन은 신체의 다양한 활동을 조절하는 역할을 하며 어두운 환경에서는 신체가 활발한 활동을 할 수 없기 때문에 호르모ンの 수치는 낮아집니다 또한 어둠은 신경계의 활동을 억제시켜서 잠을 자도록 유도합니다 새벽에는 기압의 변화로 인해 호흡과 혈액순환 신체 활동 등이 활발하게 이루어지지 않기 때문에 잠이 자는 시간이 많습니다 이러한 이유로 새벽에 자는 것이 일반적입니다 하지만 개인마다의 차이가 있을 수 있으므로 개인적으로 적합한 잠의 시간을 맞추는 것이 좋습니다 예를 들면 일주일에 7 8시간 정도의 수면이 적절한 것으로 알려져 있습니다 따라서 개인적인 수면 습관을 조정하여 적정 수면량을 유지하는 것이 중요합니다 이외에도 수면 환경 식사 운동 스트레스

정답 원본: 어두운 환경에서는 멜라토닌 호르몬 분비량이 증가하여 잠을 자게 됩니다. 멜라토닌은 생체리듬을 주관하는 작용을 지니며, 망막에 도달하는 빛의 양에 따라 분비량이 조절됩니다. 또한, 멜라토닌 제가 수면 장애와 시차로 인한 피로회복에 탁월한 효과를 발휘하


질문: 일본의 일촌일품이란 정책은 무엇인가요?
정답: 일촌일품은 일본 지방자치제도에서 각 지자체가 자신들만의 특화상품을 개발하여 지역경제에 기여하는 제도입니다. 이를 통해 관광객을 유치하고 지역 상권을 활성화시키는 등 긍정적인 영향을 미칠 수 있습니다. 특산품을 상품화하여 한 지역에만 집중적으로 판매하는 경우도 많지만, 대형 유원지나 교육시설을 개발하여 이를 주요 명소로 만드는 경우도 있습니다. 이에 따라 일촌일품의 주제는 상당히 폭넓고 다양합니다. 일본의 지리적 특성과 기후 차이를 이용하여 개발하는 경우가 많은데, 이를 흉내내어 우리나라 지자체도 지역경제 활성화를 위해 다양한 축제와 행사를 개최하는 경우가 많습니다. 하지만 일촌일품이 실패하는 경우 지자체는 부채에 시달리는 경우도 있으므로 신중한 기획과 검토가 필요합니다.
예측: 일본은 2019년 4월 1일부터 2022년 3월 31일까지 3년간 일본에서 거주하는 외국인 1,000명에 한정하여, 1년간 1인당 100만 원을 지원하는 일촌 일품 정책을 도입했습니다. 이 정책은 일본 내에 거주하고 있는 외국인들이 일본 사회에 적극적으로 참여할 수 있도록 하기 위해 마련된 정책입니다. 이와 관련하여, 일본 정부는 2020년 12월 28일, 2021년 1월 4일부터 시작된 이 정책을 연장하여 2024년 6월 30일까지 연장되었습니다. 이에 따라, 일본에서 1년에 1 000명이 넘는 외국인에게는 2 000만 원의 지원을 제공합니다. 이 외에도, 이 정책에 참여한 외국인들은 일본 내에서 5년간 거주하더라도, 3 000 000 원의 연간 지원을 받을 수 있습니다. 이외에도, 일본 내 5 000년 이상 거주한 외국인이 되면, 10 000.

질문: 냉장고, 전기밥솥의 에너지소비효율등급 기준이 강화된다고 하던데, 사실인가요?
정답: 네, 맞습니다. 산업통상자원부에서는 1 2 등급 비중이 과도해 지는 냉장고, 전기밥솥, 공기청정기, 냉온수기 등 4개 제품의 에너지소비효율등급 기준을 상향 조정하였습니다. 전기냉장고와 전기밥솥은 